# Diffusion forward and reverse snapshots

Separate known forward corruption, noise-prediction training, and iterative reverse sampling using a tiny prepared MNIST model.

**Classroom pattern:** predict first, run one cell, inspect the intermediate state, then change one parameter. All random seeds are fixed for the first run.

## Before you run

- Use the environment from `README.md`; no lecturer-specific Conda environment is required.
- Set `LECTURE2_QUICK=1` before starting Jupyter for the CPU-safe verification path.
- CUDA is selected first, then Apple MPS, then CPU. A separate MLX installation is not required.
- Read the prediction prompt before each experiment. The fixed-seed result is a mechanism check, not a quality benchmark.
- Every generated file stays inside `fallbacks/` or `checkpoints/`. No cell uploads or publishes anything.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "demo_models.py").exists():
    candidate = ROOT / "lecture 2" / "demos"
    if candidate.exists():
        ROOT = candidate
sys.path.insert(0, str(ROOT))
(ROOT / "fallbacks").mkdir(exist_ok=True)
print(f"Demo root: {ROOT}")

import numpy as np
import matplotlib.pyplot as plt
import torch
from demo_models import ConditionalTinyNoisePredictor, TinyNoisePredictor, diffusion_schedule
torch.manual_seed(11)

## 1. Load fixed data, schedule, and checkpoint

The checkpoint is intentionally small and pedagogical. It demonstrates the mechanics, not state-of-the-art image quality.

In [ ]:
checkpoint = torch.load(ROOT / "checkpoints" / "diffusion_mnist_tiny.pt", map_location="cpu", weights_only=True)
model = TinyNoisePredictor()
model.load_state_dict(checkpoint["state_dict"])
model.eval()
schedule = diffusion_schedule(checkpoint["steps"])
betas, alphas, alpha_bars = schedule["betas"], schedule["alphas"], schedule["alpha_bars"]
demo = np.load(ROOT / "data" / "mnist_demo_samples.npz")
x0 = torch.tensor(demo["images"][8:9], dtype=torch.float32) * 2.0 - 1.0
print("last training noise MSE:", round(checkpoint["history"]["mse"][-1], 4))

## 2. The known forward process

For any selected t, q(xₜ|x₀) can be sampled directly. The same fixed ε makes the signal-to-noise progression easy to compare.

In [ ]:
timesteps = [0, 15, 35, 60, 99]
generator = torch.Generator().manual_seed(5)
epsilon = torch.randn(x0.shape, generator=generator)
forward = []
for t in timesteps:
    a_bar = alpha_bars[t]
    xt = a_bar.sqrt() * x0 + (1 - a_bar).sqrt() * epsilon
    forward.append(xt)

fig, axes = plt.subplots(1, len(timesteps), figsize=(10, 2.2))
for ax, t, xt in zip(axes, timesteps, forward):
    ax.imshow(xt[0, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"t={t}\nᾱ={alpha_bars[t]:.2f}")
    ax.axis("off")
fig.suptitle("Forward corruption is known; no neural network is required")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / "05_diffusion_forward.png", dpi=180, bbox_inches="tight")
plt.show()

## 3. The training target at one random timestep

The network receives xₜ and t, then predicts the injected ε.

In [ ]:
t = torch.tensor([60])
a_bar = alpha_bars[t][:, None, None, None]
xt = a_bar.sqrt() * x0 + (1 - a_bar).sqrt() * epsilon
with torch.no_grad():
    epsilon_hat = model(xt, t)
print("epsilon-prediction MSE for this example:", torch.mean((epsilon_hat - epsilon) ** 2).item())

## 4. Iterative reverse sampling

Start from Gaussian noise. Each step predicts noise, removes part of it, and adds the variance required by the reverse transition. One trajectory is illustrative evidence, not a quality evaluation.

In [ ]:
@torch.no_grad()
def reverse_sample(seed=0):
    generator = torch.Generator().manual_seed(seed)
    x = torch.randn((1, 1, 28, 28), generator=generator)
    capture = {99, 75, 50, 25, 0}
    snapshots = {}
    for t_index in reversed(range(len(betas))):
        t = torch.tensor([t_index])
        eps_hat = model(x, t)
        alpha = alphas[t_index]
        alpha_bar = alpha_bars[t_index]
        beta = betas[t_index]
        mean = (x - (beta / torch.sqrt(1 - alpha_bar)) * eps_hat) / torch.sqrt(alpha)
        if t_index > 0:
            noise = torch.randn(x.shape, generator=generator)
            previous_alpha_bar = alpha_bars[t_index - 1]
            posterior_variance = beta * (1 - previous_alpha_bar) / (1 - alpha_bar)
            x = mean + torch.sqrt(posterior_variance) * noise
        else:
            x = mean
        if t_index in capture:
            snapshots[t_index] = x.clamp(-1, 1).clone()
    return snapshots

reverse = reverse_sample()
shown = [99, 75, 50, 25, 0]
fig, axes = plt.subplots(1, len(shown), figsize=(10, 2.2))
for ax, t, image in zip(axes, shown, [reverse[t] for t in shown]):
    ax.imshow(image[0, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"after t={t}")
    ax.axis("off")
fig.suptitle("The learned reverse transition gradually exposes structure")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / "05_diffusion_reverse.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Classifier-free guidance

A second model was trained with class labels randomly removed. At sampling time it predicts noise twice:

ε_cfg = ε_uncond + w(ε_cond − ε_uncond).

The difference points toward features associated with the requested digit. Larger guidance usually improves class compliance but can reduce diversity and amplify artifacts.

In [ ]:
cfg_checkpoint = torch.load(ROOT / "checkpoints" / "diffusion_mnist_cfg.pt", map_location="cpu", weights_only=True)
cfg_model = ConditionalTinyNoisePredictor(num_classes=cfg_checkpoint["num_classes"])
cfg_model.load_state_dict(cfg_checkpoint["state_dict"])
cfg_model.eval()
print("CFG training MSE:", round(cfg_checkpoint["history"]["mse"][-1], 4))
print("null label:", cfg_model.null_label, "condition dropout:", cfg_checkpoint["condition_dropout"])

@torch.no_grad()
def guided_reverse(initial, start_t, digit, guidance_scale=3.0, seed=31, capture=False):
    generator = torch.Generator().manual_seed(seed)
    x = initial.clone()
    snapshots = {}
    label = torch.tensor([digit])
    null = torch.tensor([cfg_model.null_label])
    for t_index in reversed(range(start_t + 1)):
        t = torch.tensor([t_index])
        eps_unconditional = cfg_model(x, t, null)
        eps_conditional = cfg_model(x, t, label)
        eps_guided = eps_unconditional + guidance_scale * (eps_conditional - eps_unconditional)
        alpha, alpha_bar, beta = alphas[t_index], alpha_bars[t_index], betas[t_index]
        mean = (x - beta / torch.sqrt(1 - alpha_bar) * eps_guided) / torch.sqrt(alpha)
        if t_index == 0:
            x = mean
        else:
            previous_alpha_bar = alpha_bars[t_index - 1]
            posterior_variance = beta * (1 - previous_alpha_bar) / (1 - alpha_bar)
            x = mean + torch.sqrt(posterior_variance) * torch.randn(x.shape, generator=generator)
        if capture and t_index in {start_t, 75, 50, 25, 0}:
            snapshots[t_index] = x.clamp(-1, 1).clone()
    return x.clamp(-1, 1), snapshots

## 6. Compare guidance strengths from the same initial noise

Every column starts from exactly the same initial noise and reuses the same random noise at every reverse step. Changes can therefore be attributed to guidance rather than the random path. We use the DDPM posterior variance, which is smaller than simply adding √βₜ noise.

In [ ]:
target_digit = 3
guidance_scales = [0.0, 1.0, 2.0, 3.0]
initial_generator = torch.Generator().manual_seed(27)
initial_noise = torch.randn((1, 1, 28, 28), generator=initial_generator)
guided_results = [guided_reverse(initial_noise, 99, target_digit, scale, seed=91)[0] for scale in guidance_scales]

fig, axes = plt.subplots(1, len(guidance_scales), figsize=(8.5, 2.3))
for ax, scale, image in zip(axes, guidance_scales, guided_results):
    ax.imshow(image[0, 0], cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"w={scale:g}")
    ax.axis("off")
fig.suptitle(f"Same noise, requested digit {target_digit}: guidance changes class evidence")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / "05_diffusion_cfg_scales.png", dpi=180, bbox_inches="tight")
plt.show()

## 7. Guided refinement from an intermediate noise level

Starting from pure noise tests unconditional generation. Starting from a partly corrupted real image tests guided denoising or image-to-image refinement. The latter is easier because some source structure remains; it should not be reported as pure generation.

In [ ]:
source_digit = 3
source = torch.tensor(demo["images"][source_digit:source_digit+1], dtype=torch.float32) * 2 - 1
start_t = 60
refine_generator = torch.Generator().manual_seed(8)
refine_noise = torch.randn(source.shape, generator=refine_generator)
noised = alpha_bars[start_t].sqrt() * source + (1 - alpha_bars[start_t]).sqrt() * refine_noise
refined, refine_trace = guided_reverse(noised, start_t, source_digit, guidance_scale=3.0, seed=52, capture=True)

fig, axes = plt.subplots(1, 3, figsize=(6.6, 2.3))
for ax, image, title in zip(axes, [source, noised, refined], ["source", f"noised to t={start_t}", "guided reverse"]):
    ax.imshow(image[0, 0], cmap="gray", vmin=-1, vmax=1)
    ax.set_title(title); ax.axis("off")
fig.suptitle("Guided refinement preserves information that pure generation never received")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / "05_diffusion_guided_refinement.png", dpi=180, bbox_inches="tight")
plt.show()

## Interpretation check and optional extension

Distinguish forward corruption, the supervised noise target, unconditional reverse sampling, classifier-free guidance, and guided refinement. State which outputs count as pure generation.

**Optional extension:** hold the initial noise fixed and plot class compliance and diversity across guidance strengths.

Record the changed parameter, the fixed controls, the observed evidence, and one limitation.